<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Подготовка" data-toc-modified-id="Подготовка-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка</a></span></li><li><span><a href="#Обучение" data-toc-modified-id="Обучение-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Обучение</a></span></li><li><span><a href="#Выводы" data-toc-modified-id="Выводы-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Выводы</a></span></li><li><span><a href="#Чек-лист-проверки" data-toc-modified-id="Чек-лист-проверки-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Чек-лист проверки</a></span></li></ul></div>

# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75. 

**Инструкция по выполнению проекта**

1. Загрузите и подготовьте данные.
2. Обучите разные модели. 
3. Сделайте выводы.

Для выполнения проекта применять *BERT* необязательно, но вы можете попробовать.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

## Введение

Целью проекта является выбрать и обучить модель для классификации комментариев (позитивные/негативные) со значением метрики качества F1 не менее 0.75.

При выполнении проекта сравним модели с помощью TF-IDF, c помощью языковых представлений без и с использованием BERT.

### План работы

Работа будет выполняться согласно следующего плана:
- загрузка и предобработка данных
- подготовка данных для обучения моделей
- обучение моделей с различными гиперпараметрами и выбор оптимальной методом кросс-валидации
- тестирование модели на тестовой выборке и оценка ее адекватности

In [1]:
!pip install spacy
!spacy download en
!pip install gensim

⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 17.6 MB/s eta 0:00:0000:0100:01


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import os

import pandas as pd
import numpy as np
import re

import spacy
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import gensim.downloader as api
from gensim.models import Word2Vec

from sklearn.experimental import enable_halving_search_cv
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, HalvingRandomSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

import lightgbm as lgb

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/dmitry/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/dmitry/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Подготовка

### Загрузка данных

In [3]:
local_path = 'toxic_comments.csv'
server_path = '/datasets/toxic_comments.csv'

In [4]:
if os.path.exists(local_path):
    data = pd.read_csv(local_path)
else:
    data = pd.read_csv(server_path)

### Предобработка данных

Для начала, рассмотрим имеющийся датасет.

In [5]:
print(data.info())
data.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159292 entries, 0 to 159291
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  159292 non-null  int64 
 1   text        159292 non-null  object
 2   toxic       159292 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.6+ MB
None


,Unnamed: 0,text,toxic
0,0,Explanation\nWhy the edits made under my usern...,0
1,1,D'aww! He matches this background colour I'm s...,0
2,2,"Hey man, I'm really not trying to edit war. It...",0
3,3,"""\nMore\nI can't make any real suggestions on ...",0
4,4,"You, sir, are my hero. Any chance you remember...",0
5,5,"""\n\nCongratulations from me as well, use the ...",0
6,6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1
7,7,Your vandalism to the Matt Shirvington article...,0
8,8,Sorry if the word 'nonsense' was offensive to ...,0
9,9,alignment on this subject and which are contra...,0


Как видно из представленной выше информации, в датасете есть три столбца - `Unnamed: 0`, повторяющий индекс, `text` - текст комментария, `toxic` - категориальный признак, описывающий, является ли комментарий негативным.
Всего в датасете представлено почти 160 000 объектов.
Пропуски в датасете отсутствуют.

Проверим датасет на наличие дубликатов.

In [6]:
data.duplicated().sum()

0

Дубликатов нет.

Удалим ненужный столбец, повторяющий индекс.

In [7]:
data = data.drop('Unnamed: 0', axis=1)

### Подготовка данных для обучения модели 

Начнем с подготовки текстов для дальнейшего обучения.
Для начала очистим текст от знаков препинания и стоп-слов, а так же лемматизируем имеющиеся в тексте слова.

In [8]:
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

In [9]:
del_n = re.compile('\n')
clean_text = re.compile("(?!')[\W\d]+|(?![\w])' ")

stopwords_list = list(stopwords.words('english'))

def clean(text):
    text = del_n.sub(' ', str(text).lower())
    res_text = clean_text.sub(' ', text)
    return res_text

def del_stopwords(text):
    clean_tokens = tuple(
        map( lambda x: x if x not in stopwords_list else '', word_tokenize(text) )
    )
    res_text = ' '.join(clean_tokens)
    return res_text

def lemmatize(text):    
    doc = nlp(text)
    lemms = ' '.join([token.lemma_ for token in doc])
    return lemms

def prepare(text):
    clean_text = clean(text)
    no_sw_text = del_stopwords(clean_text)
    lemm_text = lemmatize(no_sw_text)
    return lemm_text

In [11]:
data['text_lemms'] = data['text'].apply(prepare)

In [12]:
data.head(10)

,text,toxic,text_lemms
0,Explanation\nWhy the edits made under my usern...,0,explanation edit make username hardcore ...
1,D'aww! He matches this background colour I'm s...,0,d'aww match background colour ' m seemin...
2,"Hey man, I'm really not trying to edit war. It...",0,hey man ' m really try edit war 's ...
3,"""\nMore\nI can't make any real suggestions on ...",0,can not make real suggestion improvemen...
4,"You, sir, are my hero. Any chance you remember...",0,sir hero chance remember page 's
5,"""\n\nCongratulations from me as well, use the ...",0,congratulation well use tool well talk
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,cocksucker piss around work
7,Your vandalism to the Matt Shirvington article...,0,vandalism matt shirvington article rev...
8,Sorry if the word 'nonsense' was offensive to ...,0,sorry word nonsense offensive anyway ...
9,alignment on this subject and which are contra...,0,alignment subject contrary dulithgow


Выделим целевой признак и признак, по которому будет проводиться обучение и дальнейшее предсказывание.

In [13]:
features = data['text_lemms']
target = data['toxic']

Разделим выборку на тренировочную и тестовую.

In [14]:
features_train, features_test, target_train, target_test =\
train_test_split(features, target, test_size = 0.25)

#### Подготовка текстов для модели линейной регрессии 

#### Подготовка текстов для модели решающего дерева и градиентного бустинга

Для вышеупомянутых моделей используем библиотеку word2vec для векторизации текстов. Судя по первым 10 комментариям, лучше всего использовать модель, обученную на постах в твиттере.


## Обучение

#### Модель логистической регрессии

Подготовим pipeline для предобработки текстов с помощью TF-IDF и дальнейшего обучения модели логистической регрессии, чтобы обеспечить выбор оптимальных параметров для TF-IDF.

In [15]:
RANDOM_STATE = 12345

In [47]:
log_reg_pipeline = Pipeline(
    steps = [('tfidf', TfidfVectorizer() ),('log_reg',LogisticRegression() )]
)

log_reg_param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2), (2,2), (1,3), (2,3)],
    'tfidf__analyzer': ['word', 'char', 'char_wb'],
    'tfidf__sublinear_tf': [False, True],
    'log_reg__penalty': ['l2', None],
    'log_reg__C': [0.00001, 0.1, 10.0, 100.0 1000.0, 100000.0],
    "log_reg__max_iter": [10000000],
    "log_reg__random_state": [RANDOM_STATE]
}

In [55]:
log_reg_best_model = RandomizedSearchCV(
    log_reg_pipeline,
    log_reg_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE
)

In [56]:
log_reg_best_model.fit(features_train, target_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will

[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(2, 2), tfidf__sublinear_tf=True; total time=   0.0s
[CV] END log_

/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data 

[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(2, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_

...

/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1181: UserWarning: Setting penalty=None will

[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(2, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(2, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(2, 2), tfidf__sublinear_tf=True; total time=   0.0s
[CV] END lo

[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=word, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=char, tfidf__ngram_range=(2, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=char, tfidf__ngram_range=(2, 2), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=char, tfidf__ngram_range=(1, 3), tfidf__sublinear_tf=False; total time=   0.0s
[CV] END log_reg__C=1e-05, log_reg__max_iter=100000, log_reg__penalty=l2, log_reg__random_state=12345, tfidf__analyzer=char, tfidf__ngram_range=(1, 3), tfidf__sublinear_tf=True; total time=   0.0s
[CV] END lo

/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/dmitry/opt/anaconda3/envs/ds_practicum_env/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
   

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                             ('log_reg',
                                              LogisticRegression())]),
                   n_jobs=-1,
                   param_distributions={'log_reg__C': [1e-05, 0.1, 10.0, 1000.0,
                                                       100000.0],
                                        'log_reg__max_iter': [100000],
                                        'log_reg__penalty': ['l2', None],
                                        'log_reg__random_state': [12345],
                                        'tfidf__analyzer': ['word', 'char',
                                                            'char_wb'],
                                        'tfidf__ngram_range': [(1, 1), (1, 2),
                                                               (2, 2), (1, 3),
                                                               (2, 3)],
                                        'tfidf__sublinear_tf': [False, True]},
                   random_state=12345, scoring='f1', verbose=1)

In [57]:
log_reg_best_model.best_score_

0.7834005816883797

Модель логистической регрессии показала значение метрики F1, превышающее пороговое значение (0.78)

#### Модель случайного дерева

Гиперпараметры случайного дерева будем подбирать с помощью RandomizedSearchCV.
При обучении и предсказании будет использоваться корпус текстов, приведенных в векторный вид с помощью модели word2vec `glove twitter 50`.

In [58]:
wv_model = api.load('glove-twitter-50')

In [59]:
words = set(wv_model.index_to_key )
 
wv_corpus_train = np.array([np.array([wv_model[i] for i in ls if i in words])
                         for ls in features_train])
wv_corpus_test = np.array([np.array([wv_model[i] for i in ls if i in words])
                         for ls in features_test])


wv_corpus_train_avg = []
for v in wv_corpus_train:
    if v.size:
        wv_corpus_train_avg.append(v.mean(axis=0))
    else:
        wv_corpus_train_avg.append(np.zeros(50, dtype=float))
        
wv_corpus_test_avg = []
for v in wv_corpus_test:
    if v.size:
        wv_corpus_test_avg.append(v.mean(axis=0))
    else:
        wv_corpus_test_avg.append(np.zeros(50, dtype=float))

/var/folders/gb/njsrwj253nd14gv4htvqf5pc0000gn/T/ipykernel_18851/931219301.py:3: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  wv_corpus_train = np.array([np.array([wv_model[i] for i in ls if i in words])
/var/folders/gb/njsrwj253nd14gv4htvqf5pc0000gn/T/ipykernel_18851/931219301.py:5: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  wv_corpus_test = np.array([np.array([wv_model[i] for i in ls if i in words])


In [60]:
random_forest_model = RandomForestClassifier(random_state=RANDOM_STATE)

forest_model_params = {
    'n_estimators' : range(10,50,10),
    'max_depth' : [None] + [i for i in range(2, 7)],
    'min_samples_split' : range(2, 11, 5),
    'min_samples_leaf' : range(1,11,5),
    'max_features' : [1, 'sqrt', 'log2']
}

best_forest_model = RandomizedSearchCV(
    random_forest_model,
    forest_model_params,
    n_iter=50,
    scoring='f1',
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_STATE
)

In [61]:
best_forest_model.fit(wv_corpus_train_avg, target_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(estimator=RandomForestClassifier(random_state=12345),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'max_depth': [None, 2, 3, 4, 5, 6],
                                        'max_features': [1, 'sqrt', 'log2'],
                                        'min_samples_leaf': range(1, 11, 5),
                                        'min_samples_split': range(2, 11, 5),
                                        'n_estimators': range(10, 50, 10)},
                   random_state=12345, scoring='f1', verbose=2)

In [62]:
best_forest_model.best_score_

0.21944087814598817

Модель случайного леса на преобразованных в векторы словах показала гораздо более худший результат, чем пороговое значение и модель логистической регрессии.

#### Модель градиентного бустинга LightGBM 

В качестве еще одной модели рассмотрим модель градиентного бустинга LightGBM.

In [63]:
lgb_model = lgb.LGBMClassifier(
    random_state=RANDOM_STATE
)

lgb_params = {
    'num_leaves' : range(10,100,10),
    'max_depth' : range(7,15,1),
    'n_estimators' : range(100,1000,100),
}

best_lgb_model = RandomizedSearchCV(
    lgb_model,
    lgb_params,
    n_iter=50,
    scoring='f1',
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_STATE
)

[CV] END max_depth=2, max_features=sqrt, min_samples_leaf=1, min_samples_split=7, n_estimators=40; total time=   4.6s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=40; total time=  29.0s
[CV] END max_depth=3, max_features=sqrt, min_samples_leaf=6, min_samples_split=2, n_estimators=30; total time=   4.9s
[CV] END max_depth=2, max_features=1, min_samples_leaf=1, min_samples_split=2, n_estimators=30; total time=   0.7s
[CV] END max_depth=3, max_features=1, min_samples_leaf=1, min_samples_split=2, n_estimators=20; total time=   0.7s
[CV] END max_depth=3, max_features=1, min_samples_leaf=1, min_samples_split=2, n_estimators=20; total time=   0.7s
[CV] END max_depth=4, max_features=sqrt, min_samples_leaf=6, min_samples_split=2, n_estimators=30; total time=   7.2s
[CV] END max_depth=4, max_features=sqrt, min_samples_leaf=6, min_samples_split=2, n_estimators=30; total time=   7.2s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=6, m

[CV] END max_depth=4, max_features=1, min_samples_leaf=6, min_samples_split=7, n_estimators=40; total time=   1.6s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=7, n_estimators=20; total time=   5.2s
[CV] END max_depth=5, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=40; total time=   8.5s
[CV] END max_depth=4, max_features=1, min_samples_leaf=6, min_samples_split=7, n_estimators=20; total time=   1.0s
[CV] END max_depth=4, max_features=1, min_samples_leaf=6, min_samples_split=7, n_estimators=20; total time=   1.0s
[CV] END max_depth=4, max_features=1, min_samples_leaf=6, min_samples_split=7, n_estimators=20; total time=   1.0s
[CV] END max_depth=4, max_features=1, min_samples_leaf=6, min_samples_split=7, n_estimators=20; total time=   1.0s
[CV] END max_depth=4, max_features=1, min_samples_leaf=6, min_samples_split=7, n_estimators=20; total time=   1.0s
[CV] END max_depth=3, max_features=sqrt, min_samples_leaf=6, min_samples_s

[CV] END max_depth=2, max_features=sqrt, min_samples_leaf=1, min_samples_split=7, n_estimators=40; total time=   4.4s
[CV] END max_depth=4, max_features=1, min_samples_leaf=6, min_samples_split=7, n_estimators=40; total time=   1.6s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=40; total time=  28.2s
[CV] END max_depth=3, max_features=sqrt, min_samples_leaf=6, min_samples_split=2, n_estimators=30; total time=   5.3s
[CV] END max_depth=2, max_features=1, min_samples_leaf=1, min_samples_split=2, n_estimators=30; total time=   0.7s
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=6, min_samples_split=7, n_estimators=30; total time=  24.8s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=6, min_samples_split=2, n_estimators=30; total time=   7.8s
[CV] END max_depth=3, max_features=1, min_samples_leaf=6, min_samples_split=7, n_estimators=30; total time=   1.0s
[CV] END max_depth=3, max_features=1, min_samples_leaf=6, m

In [64]:
best_lgb_model.fit(wv_corpus_train_avg, target_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(estimator=LGBMClassifier(random_state=12345), n_iter=50,
                   n_jobs=-1,
                   param_distributions={'max_depth': range(7, 15),
                                        'n_estimators': range(100, 1000, 100),
                                        'num_leaves': range(10, 100, 10)},
                   random_state=12345, scoring='f1', verbose=2)

In [66]:
best_lgb_model.best_score_

0.29397560093961794

Модель градиентного бустинга LightGBM показала результат, лучший чем модель решающего дерева, но гораздо ниже логистической регрессии и порогового значения.

#### Вывод по итогам обучения

По итогам обучения была выбрана модель линейной регрессии с предобработкой текстов с помощью TF-IDF.
Значение метрики F1 для данной связки составило 0.78, что является самой высокой из всех проанализированных моделей и превосходит пороговое значение.

### Тестирование модели

In [69]:
best_model = log_reg_best_model.best_estimator_

In [70]:
predictions = best_model.predict(features_test)

In [71]:
f1_log_reg = f1_score(target_test, predictions)

print(f'Значение метрики F1 наилучшей модели для тестовой выборки составило: {f1_log_reg}')

Значение метрики F1 наилучшей модели для тестовой выборки составило: 0.7906358073955838


## Выводы

В результате выполненной работы, лучшей моделью оказалась связка из обработки текстов с помощью TF-IDF и модели логистической регресии. Кроме нее, были рассмотрены модели случайного леса и LightGBM, обучение которых проходило на текстах, преобразованных в векторы с помощью модели, обученной на постах Твиттера.

На тестовой выборке лучшая модель показала значение метрики F1, равное 0.79.

## Чек-лист проверки

- [x]  Jupyter Notebook открыт
- [x]  Весь код выполняется без ошибок
- [x]  Ячейки с кодом расположены в порядке исполнения
- [x]  Данные загружены и подготовлены
- [x]  Модели обучены
- [x]  Значение метрики *F1* не меньше 0.75
- [x]  Выводы написаны